# Product Data Quality Analysis

This notebook inspects the generated DuckDB product pipeline outputs. Run `make run` before opening it.

In [ ]:
import csv
import json
import subprocess
from pathlib import Path

DB = Path('../product_pipeline.duckdb')
REPORTS = Path('../reports')
OUTPUT = Path('../output')

def read_csv(path):
    with path.open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

def duckdb_json(sql):
    result = subprocess.run(
        ['duckdb', '-json', str(DB), '-c', sql],
        check=True,
        text=True,
        stdout=subprocess.PIPE,
    )
    return json.loads(result.stdout)

quality_rows = read_csv(REPORTS / 'quality_report.csv')
error_rows = read_csv(REPORTS / 'product_errors.csv')
quality_rows

## Quality Summary

In [ ]:
total_errors = sum(int(row['error_count']) for row in quality_rows)
failing_checks = [row for row in quality_rows if int(row['error_count']) > 0]

{
    'total_errors': total_errors,
    'failing_check_count': len(failing_checks),
    'error_row_count': len(error_rows),
}

In [ ]:
for row in sorted(quality_rows, key=lambda item: int(item['error_count']), reverse=True):
    count = int(row['error_count'])
    bar = '#' * count
    print(f"{row['check_name']:<22} {count:>3} {bar}")

## Error Row Sample

In [ ]:
for row in error_rows[:10]:
    print(
        row['jan_code'],
        row['product_name'],
        row['maker_name'],
        row['error_reasons'],
        sep=' | ',
    )

## Parquet Dataset Analysis

In [ ]:
duckdb_json("""
SELECT
  category_name,
  count(*) AS product_count,
  avg(price) AS avg_price
FROM read_parquet('../output/products_by_category/**/*.parquet', hive_partitioning = true)
GROUP BY category_name
ORDER BY category_name;
""")

In [ ]:
duckdb_json("""
SELECT
  ingest_date,
  category_name,
  count(*) AS product_count
FROM read_parquet('../output/products_by_ingest_date/**/*.parquet', hive_partitioning = true)
GROUP BY ingest_date, category_name
ORDER BY ingest_date, category_name;
""")

In [ ]:
duckdb_json("""
SELECT
  maker_name,
  count(*) AS product_count,
  min(price) AS min_price,
  max(price) AS max_price
FROM read_parquet('../output/products.parquet')
GROUP BY maker_name
ORDER BY product_count DESC, maker_name;
""")